# PlantCLEF 2015 Test LeafScan Archive

Downloads the official PlantCLEF 2015 annotated test package, extracts LeafScan metadata/images, and stores a compact archive on Google Drive for evaluation notebooks.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 2. Clone Or Update Project

In [4]:
from pathlib import Path
import os
import shutil
import subprocess

PROJECT_DIR = Path('/content/diploma')
REPO_URL = 'https://github.com/robodanill/diploma.git'
BRANCH = 'robodanill/main'


def clone_project():
    os.chdir('/content')
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)


def pull_project() -> bool:
    if not (PROJECT_DIR / '.git').exists():
        return False
    result = subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT_DIR)
    return result.returncode == 0


if PROJECT_DIR.exists():
    print(f'Trying to update existing project: {PROJECT_DIR}')
    if not pull_project():
        print('Pull failed or project is not a git repository; cloning a fresh copy.')
        clone_project()
else:
    print(f'Project not found at {PROJECT_DIR}; cloning a fresh copy.')
    clone_project()

os.chdir(PROJECT_DIR)
subprocess.run(['python', '-m', 'pip', 'install', '-e', '.[ml]'], check=True)
commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=PROJECT_DIR, text=True).strip()
print(f'Project commit: {commit}')


Trying to update existing project: /content/diploma
Project commit: c520dcd


## 3. Download Official Test Package

In [5]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1

RAW_DIR=/content/plantclef2015_test_raw
LOCAL_RAW=/content/PlantCLEF2015TestDataWithAnnotations.tar.gz
DRIVE_RAW=/content/drive/MyDrive/PlantCLEF2015TestDataWithAnnotations.tar.gz
URL=https://lab.plantnet.org/LifeCLEF/PlantCLEF2015/TestPackage/PlantCLEF2015TestDataWithAnnotations.tar.gz

mkdir -p /content/drive/MyDrive
if [ -s "$DRIVE_RAW" ] && tar -tzf "$DRIVE_RAW" >/dev/null 2>&1; then
  echo "using existing validated Drive raw archive"
  cp "$DRIVE_RAW" "$LOCAL_RAW"
else
  echo "downloading raw test package to local Colab disk, not directly to Google Drive"
  wget -c --tries=20 --timeout=120 --read-timeout=120 "$URL" -O "$LOCAL_RAW"
fi
tar -tzf "$LOCAL_RAW" >/dev/null
rm -rf "$RAW_DIR"
mkdir -p "$RAW_DIR"
tar -xzf "$LOCAL_RAW" -C "$RAW_DIR"
find "$RAW_DIR" -type f | wc -l
find "$RAW_DIR" -type f \( -iname '*.jpg' -o -iname '*.jpeg' -o -iname '*.png' \) | wc -l
find "$RAW_DIR" -type f -iname '*.xml' | wc -l


downloading raw test package to local Colab disk, not directly to Google Drive
42892
21446
21446


--2026-05-03 16:44:50--  https://lab.plantnet.org/LifeCLEF/PlantCLEF2015/TestPackage/PlantCLEF2015TestDataWithAnnotations.tar.gz
Resolving lab.plantnet.org (lab.plantnet.org)... 193.51.117.136
Connecting to lab.plantnet.org (lab.plantnet.org)|193.51.117.136|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4505266619 (4.2G) [application/x-gzip]
Saving to: ‘/content/PlantCLEF2015TestDataWithAnnotations.tar.gz’

     0K .......... .......... .......... .......... ..........  0%  144K 8h28m
    50K .......... .......... .......... .......... ..........  0%  289K 6h21m
   100K .......... .......... .......... .......... ..........  0%  186M 4h14m
   150K .......... .......... .......... .......... ..........  0%  189M 3h10m
   200K .......... .......... .......... .......... ..........  0%  289K 3h23m
   250K .......... .......... .......... .......... ..........  0%  168M 2h49m
   300K .......... .......... .......... .......... ..........  0% 76.9M 2h25m
   350K .

## 4. Build LeafScan Test Bundle

In [6]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma

RAW_DIR=/content/plantclef2015_test_raw
WORK_DIR=/content/plantclef2015_test_leafscan_bundle
RAW_METADATA=/content/plantclef2015_test_metadata_raw.csv
DRIVE_BUNDLE=/content/drive/MyDrive/PlantCLEF2015_leafscan_test.tar.gz

rm -rf "$WORK_DIR"
mkdir -p "$WORK_DIR"

python -u -m plant_classifier.data.plantclef_cli \
  --source-root "$RAW_DIR" \
  --image-root "$RAW_DIR" \
  --relative-to "$RAW_DIR" \
  --output "$RAW_METADATA" \
  --content LeafScan

python - <<'PY_COPY_LEAFSCAN'
import csv
import shutil
from collections import Counter
from pathlib import Path

raw_root = Path('/content/plantclef2015_test_raw')
raw_metadata = Path('/content/plantclef2015_test_metadata_raw.csv')
bundle = Path('/content/plantclef2015_test_leafscan_bundle')
image_root = bundle / 'leafscan'
metadata_out = image_root / 'metadata.csv'

with raw_metadata.open(newline='', encoding='utf-8') as file:
    rows = list(csv.DictReader(file))

metadata_out.parent.mkdir(parents=True, exist_ok=True)
image_root.mkdir(parents=True, exist_ok=True)

fieldnames = ['image_path', 'family', 'genus', 'species', 'content', 'split', 'source_xml']
with metadata_out.open('w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    for row in rows:
        rel_image = Path(row['image_path'])
        src = raw_root / rel_image
        dst_rel = rel_image
        dst = image_root / dst_rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        writer.writerow({
            'image_path': str(dst_rel),
            'family': row.get('family', ''),
            'genus': row.get('genus', ''),
            'species': row.get('species', ''),
            'content': row.get('content', ''),
            'split': row.get('split', 'test') or 'test',
            'source_xml': row.get('source_xml', ''),
        })

genera = len({row['genus'] for row in rows})
species = len({row['species'] for row in rows})
print(f'leafscan rows: {len(rows)}')
print('genera:', genera)
print('species:', species)
if len(rows) != 221 or genera != 43 or species != 60:
    raise RuntimeError(f'Expected PlantCLEF test LeafScan 221 rows / 43 genera / 60 species, got {len(rows)} / {genera} / {species}')
print('split counts:', Counter(row.get('split', 'test') or 'test' for row in rows))
PY_COPY_LEAFSCAN

tar -czf "$DRIVE_BUNDLE" -C "$WORK_DIR" .
ls -lh "$DRIVE_BUNDLE"
tar -tzf "$DRIVE_BUNDLE" | sed -n '1,20p'


saved 221 rows to /content/plantclef2015_test_metadata_raw.csv
leafscan rows: 221
genera: 43
species: 60
split counts: Counter({'test': 221})
-rw------- 1 root root 22M May  3 16:51 /content/drive/MyDrive/PlantCLEF2015_leafscan_test.tar.gz
./
./leafscan/
./leafscan/metadata.csv
./leafscan/PlantCLEF2015TestDataWithAnnotations/
./leafscan/PlantCLEF2015TestDataWithAnnotations/59383.jpg
./leafscan/PlantCLEF2015TestDataWithAnnotations/80369.jpg
./leafscan/PlantCLEF2015TestDataWithAnnotations/53330.jpg
./leafscan/PlantCLEF2015TestDataWithAnnotations/99296.jpg
./leafscan/PlantCLEF2015TestDataWithAnnotations/91204.jpg
./leafscan/PlantCLEF2015TestDataWithAnnotations/36728.jpg
./leafscan/PlantCLEF2015TestDataWithAnnotations/65826.jpg
./leafscan/PlantCLEF2015TestDataWithAnnotations/58072.jpg
./leafscan/PlantCLEF2015TestDataWithAnnotations/25842.jpg
./leafscan/PlantCLEF2015TestDataWithAnnotations/68556.jpg
./leafscan/PlantCLEF2015TestDataWithAnnotations/57001.jpg
./leafscan/PlantCLEF2015TestDataWi

## 5. Smoke-Extract Bundle

In [7]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1

TMP=/content/plantclef2015_test_leafscan_check
ARCHIVE=/content/drive/MyDrive/PlantCLEF2015_leafscan_test.tar.gz
rm -rf "$TMP"
mkdir -p "$TMP"
tar -xzf "$ARCHIVE" -C "$TMP"
test -f "$TMP/leafscan/metadata.csv"
python - <<'PY_CHECK_BUNDLE'
import csv
from collections import Counter
from pathlib import Path
root = Path('/content/plantclef2015_test_leafscan_check')
with (root / 'leafscan' / 'metadata.csv').open(newline='', encoding='utf-8') as file:
    rows = list(csv.DictReader(file))
missing = [row['image_path'] for row in rows if not (root / 'leafscan' / row['image_path']).exists()]
print('rows:', len(rows))
print('missing images:', len(missing))
print('content:', Counter(row.get('content', '') for row in rows))
print('split:', Counter(row.get('split', '') for row in rows))
if missing:
    raise FileNotFoundError(missing[:5])
PY_CHECK_BUNDLE


rows: 221
missing images: 0
content: Counter({'LeafScan': 221})
split: Counter({'test': 221})
